# Velocity & Backlog Projection

Velocity tracking for GitHub Project
[#2](https://github.com/orgs/greenearth-social/projects/2/views/3).

All computation lives in the `velocity` library (unit-tested); this notebook is
presentation only.

**Prerequisite:** the `gh` CLI must be authenticated with the project scope:

```
gh auth refresh -s read:project
```

**Known limitation:** the GitHub GraphQL API returns project items in the
board's manual order, which we treat as priority order. Per-view (view #3)
custom sorting is not exposed by the API.

In [ ]:
# Auto-reload the `velocity` library on edit, so you don't have to
# restart the kernel after changing library code.
%load_ext autoreload
%autoreload 2

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # repo root, so `velocity` imports

import pandas as pd
import plotly.graph_objects as go

from velocity import github_project, velocity as vel, burndown, points

items = github_project.fetch_items()
print(f"Fetched {len(items)} project items")

Fetched 439 project items


## Velocity
Average completed points per week over the last 3 weeks.

In [ ]:
v = vel.compute_velocity(items, weeks=3)
recent = vel.completed_weeks(items, weeks=3)
print(f"Velocity: {v:.1f} points / week")
for wk, pts in recent:
    print(f"  week of {wk}: {pts} pts")

Velocity: 24.0 points / week
  week of 2026-06-15: 26 pts
  week of 2026-06-22: 33 pts
  week of 2026-06-29: 13 pts


## Completed points per week (last ~2 months)

In [ ]:
hist = burndown.completed_history(items, weeks_back=8)
fig = go.Figure(go.Bar(x=[str(wk) for wk, _ in hist], y=[pts for _, pts in hist]))
fig.add_hline(y=v, line_dash="dash", annotation_text=f"velocity {v:.1f}")
fig.update_layout(title="Completed points per week", xaxis_title="week", yaxis_title="points")
fig.show()

## Backlog burndown
Remaining backlog points projected forward at the current velocity, down to
zero. (Weekly *completed* points are shown in the chart above.)

In [ ]:
proj = burndown.project_burndown(items, v)
total = burndown.backlog_total(items)
fig = go.Figure(go.Scatter(
    x=[str(wk) for wk, _ in proj],
    y=[rem for _, rem in proj],
    mode="lines+markers",
    name="projected remaining",
))
finish = proj[-1][0] if len(proj) > 1 else None
title = f"Backlog burndown ({total} pts" + (f", done ~{finish})" if finish else ")")
fig.update_layout(title=title, xaxis_title="week", yaxis_title="points remaining")
fig.show()

## Projected completion — next 4 weeks

Backlog tasks grouped by the week they're projected to finish, in priority
order. **`~`** marks auto-estimated points (the task is unpointed); bugs count
as 0 points.

In [ ]:
import html
from IPython.display import HTML

backlog = burndown.backlog_items(items)
assignments = burndown.assign_tasks_to_weeks(backlog, v)
grouped = burndown.upcoming_weeks(assignments, weeks=4)

TYPE_COLORS = {"bug": "#d73a49", "feature": "#28a745", "task": "#6f42c1"}
POINTS_COLOR = "#e0a52a"  # gold: real points we set, readable on light and dark
ESTIMATE_COLOR = "#8a8a8a"  # medium gray: auto-estimated points, readable on either
ROW_BORDER = "1px solid rgba(128,128,128,0.45)"  # medium gray, subtle in either mode


def render(grouped):
    out = ['<div style="font-family:-apple-system,Segoe UI,sans-serif;max-width:780px">']
    out.append('<p style="color:#888;font-size:12px;margin:0 0 8px">'
               '<b>~</b> = auto-estimated points (task is unpointed) &nbsp;&middot;&nbsp; '
               'bugs count as 0 points</p>')
    for wk, group in grouped:
        total = sum(points.normalize(i) for i in group)
        label = f"{len(group)} task" + ("" if len(group) == 1 else "s")
        out.append(
            f'<h3 style="margin:16px 0 2px">Week of {wk:%b %d}'
            f'<span style="color:#888;font-weight:normal;font-size:13px"> &mdash; {label}, {total} pts</span></h3>')
        if not group:
            out.append('<div style="color:#888;font-size:13px;padding-left:6px">(nothing projected)</div>')
            continue
        out.append('<table style="border-collapse:collapse;width:100%;font-size:13px">')
        for it in group:
            pts = points.normalize(it)
            est = points.is_estimated(it)
            pts_label = f"~{pts}" if est else str(pts)
            pts_color = ESTIMATE_COLOR if est else POINTS_COLOR
            pts_style = f"color:{pts_color};" + ("font-style:italic" if est else "")
            typ = points.effective_type(it)
            color = TYPE_COLORS.get(typ, "#888")
            badge = f'<span style="background:{color};color:#fff;border-radius:3px;padding:1px 6px;font-size:11px">{typ}</span>'
            title = html.escape(it.title)
            if it.url:
                # color:inherit -> follows the notebook theme (dark or light)
                title = f'<a href="{it.url}" style="color:inherit;text-decoration:none">{title}</a>'
            out.append(
                f'<tr style="border-top:{ROW_BORDER}">'
                f'<td style="padding:3px 8px;text-align:left;color:inherit">{title}</td>'
                f'<td style="padding:3px 8px;text-align:right;white-space:nowrap">{badge}</td>'
                f'<td style="width:44px;padding:3px 8px;text-align:right;{pts_style}">{pts_label}</td></tr>')
        out.append("</table>")
    out.append("</div>")
    return HTML("".join(out))


render(grouped)

Metrics not reported for megastream ingest,bug,0
Log the mean diversity score for each batch of returned results,feature,2
Create feed error rate dashboard & SLA (MVP),feature,2
Your Feed cold-start failure: upstream service unreachable,bug,0
Define an MVP p50 SLA for feed latency,feature,2
"Include git sha in debug feed names, deploy only clean copies",feature,~2
**** MVP demarcator ***,feature,~2
Define online eval and growth metrics,feature,~2
Support for feed transparency view in api,feature,3
Feed transparency view in frontend,feature,5
Define feed error rate SLA (GA),feature,~2
